In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import joblib
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
import tensorflow as tf

2026-01-13 11:29:25.016501: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-13 11:29:25.555192: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-13 11:29:27.366589: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
/home/joe/miniconda3/lib/python3.13/site-packages/keras/src/export/tf2onnx_lib.p

In [11]:
"""
train_regime_model.py
---------------------
Train ONE regime model with horizon as a feature.
Model type is pluggable: Gradient Boosting or Neural Net.

This script:
- Uses deseasonalized VPD anomalies
- Trains on all horizons (1–12)
- Includes horizon explicitly as an input
- Trains ONE model for ONE target-month regime
"""


# =========================
# CONFIG
# =========================
CSV_PATH = Path("full_data.csv")
DATE_COL = "Yrmo"          # YYYYMM
TARGET_COL = "VPD"        # change to "VPD" if needed

MAX_HORIZON = 12
MODEL_TYPE = "nn"          # "gb" or "nn"

REGIME_NAME = "apr_jun"
REGIME_MONTHS = {4, 5, 6}        # April, May, June

FEATURE_COLS = ["F1","F2","F3","F4","F5","F6"]        # set explicitly or auto-detect

RANDOM_STATE = 42


# =========================
# LOAD DATA
# =========================
df = pd.read_csv(CSV_PATH)

df["date"] = pd.to_datetime(df[DATE_COL].astype(str) + "01", format="%Y%m%d")
df["month"] = df["date"].dt.month
df = df.sort_values("date").reset_index(drop=True)

if TARGET_COL not in df.columns:
    raise ValueError(f"Target column '{TARGET_COL}' not found")

# =========================
# DESEASONALIZE TARGET
# =========================
monthly_mean = df.groupby("month")[TARGET_COL].mean()
df["target_anom"] = df[TARGET_COL] - df["month"].map(monthly_mean)

# =========================
# FEATURE SELECTION
# =========================
if FEATURE_COLS is None:
    exclude = {
        DATE_COL, "date", "month",
        TARGET_COL, "target_anom"
    }
    FEATURE_COLS = [
        c for c in df.columns
        if c not in exclude and pd.api.types.is_numeric_dtype(df[c])
    ]

if not FEATURE_COLS:
    raise RuntimeError("No numeric feature columns found")

print(f"Using {len(FEATURE_COLS)} features:")
print(FEATURE_COLS)

# =========================
# SCALE FEATURES & TARGET
# =========================
x_scaler = StandardScaler()
X_all = x_scaler.fit_transform(df[FEATURE_COLS].values.astype(np.float32))

y_scaler = StandardScaler()
y_all = y_scaler.fit_transform(
    df[["target_anom"]].values.astype(np.float32)
).ravel()

# =========================
# BUILD SUPERVISED PAIRS
# =========================
X_list = []
y_list = []
meta_month = []
meta_horizon = []
n = len(df)
meta_date = []
for t in range(n):
    x_t = X_all[t]

    for h in range(1, MAX_HORIZON + 1):
        j = t + h
        if j >= n:
            continue

        target_month = int(df.loc[j, "month"])
        if target_month not in REGIME_MONTHS:
            continue

        # horizon as feature (scaled)
        h_feat = np.array([h / MAX_HORIZON], dtype=np.float32)

        X_list.append(np.concatenate([x_t, h_feat]))
        y_list.append(y_all[j])
        meta_month.append(target_month)
        meta_horizon.append(h)
        meta_date.append(df.loc[j, "date"])

X = np.array(X_list, dtype=np.float32)
y = np.array(y_list, dtype=np.float32)
meta_month = np.array(meta_month)
meta_horizon = np.array(meta_horizon)
meta_date = np.array(meta_date)

print(f"\nBuilt dataset for regime '{REGIME_NAME}':")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

if len(y) < 100:
    print("⚠️ WARNING: Very few training samples for this regime")

# =========================
# TRAIN / VALIDATION SPLIT
# =========================
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    shuffle=False
)
split = int(0.8 * len(y))
meta_month_val = meta_month[split:]
meta_horizon_val = meta_horizon[split:]
val_dates = meta_date[split:]

# =========================
# MODEL FACTORY
# =========================

def quantile_loss(tau):
    """
    Quantile (pinball) loss.
    tau = 0.5 → MAE
    tau > 0.5 → penalize underprediction more
    """
    def loss(y_true, y_pred):
        e = y_true - y_pred
        return tf.reduce_mean(tf.maximum(tau * e, (tau - 1) * e))
    return loss


    
def build_two_head_nn(input_dim, tau=0.85):
    import tensorflow as tf
    from tensorflow.keras.layers import Input, Dense
    from tensorflow.keras.models import Model
    from tensorflow.keras.optimizers import Adam

    inputs = Input(shape=(input_dim,))

    # Shared representation
    x = Dense(32, activation="relu")(inputs)
    x = Dense(16, activation="relu")(x)

    # Head 1: mean prediction
    mean_out = Dense(1, name="mean")(x)

    # Head 2: upper-quantile prediction
    quantile_out = Dense(1, name="quantile")(x)

    model = Model(
        inputs=inputs,
        outputs=[mean_out, quantile_out]
    )

    model.compile(
        optimizer=Adam(1e-3),
        loss={
            "mean": "mae",
            "quantile": quantile_loss(tau)
        },
        loss_weights={
            "mean": 0.5,
            "quantile": 0.5
        }
    )

    return model


# =========================
# TRAIN MODEL
# =========================
model = build_two_head_nn(X_train.shape[1])

model.fit(
    X_train,
    {
        "mean": y_train,
        "quantile": y_train
    },
    validation_data=(
        X_val,
        {
            "mean": y_val,
            "quantile": y_val
        }
    ),
    epochs=200,
    callbacks=[EarlyStopping(patience=15, restore_best_weights=True)],
    verbose=1
)

# =========================
# SAVE ARTIFACTS
# =========================
out_prefix = f"{REGIME_NAME}_{MODEL_TYPE}"

if MODEL_TYPE == "gb":
    joblib.dump(model, f"model_{out_prefix}.pkl")
else:
    model.save(f"model_{out_prefix}.h5")

joblib.dump(x_scaler, f"x_scaler_{REGIME_NAME}.pkl")
joblib.dump(y_scaler, f"y_scaler_{REGIME_NAME}.pkl")
joblib.dump(monthly_mean, "monthly_mean.pkl")

print("\n✅ Training complete")
print(f"Saved model_{out_prefix}")


Using 6 features:
['F1', 'F2', 'F3', 'F4', 'F5', 'F6']

Built dataset for regime 'apr_jun':
X shape: (1236, 7)
y shape: (1236,)
Epoch 1/200
31/31 ━━━━━━━━━━━━━━━━━━━━ 3s 57ms/step - loss: 0.7641 - mean_loss: 1.0575 - quantile_loss: 0.4707 - val_loss: 0.6843 - val_mean_loss: 0.9920 - val_quantile_loss: 0.3641
Epoch 2/200
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.7062 - mean_loss: 1.0374 - quantile_loss: 0.3749 - val_loss: 0.6558 - val_mean_loss: 0.9981 - val_quantile_loss: 0.2989
Epoch 3/200
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.6840 - mean_loss: 1.0264 - quantile_loss: 0.3414 - val_loss: 0.6461 - val_mean_loss: 0.9958 - val_quantile_loss: 0.2814
Epoch 4/200
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.6738 - mean_loss: 1.0177 - quantile_loss: 0.3297 - val_loss: 0.6438 - val_mean_loss: 0.9939 - val_quantile_loss: 0.2790
Epoch 5/200
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.6673 - mean_loss: 1.0104 - quantile_loss: 0.3240 - val_loss: 0.6432 - val_mean_loss: 0


✅ Training complete
Saved model_apr_jun_nn


## Evaluation 

In [12]:


# ============================================================
# Predict (two-headed model)
# ============================================================
# =========================
# MODEL PREDICTION
# =========================

y_mean_pred, y_quantile_pred = model.predict(X_val)

y_mean_pred = y_mean_pred.reshape(-1)
y_quantile_pred = y_quantile_pred.reshape(-1)


# ============================================================
# Inverse-scale anomalies
# ============================================================
y_val_anom = y_scaler.inverse_transform(
    y_val.reshape(-1, 1)
).ravel()

y_mean_pred_anom = y_scaler.inverse_transform(
    y_mean_pred.reshape(-1, 1)
).ravel()

y_quantile_pred_anom = y_scaler.inverse_transform(
    y_quantile_pred.reshape(-1, 1)
).ravel()

# ============================================================
# Add monthly means back (physical VPD)
# ============================================================
monthly_mean_dict = monthly_mean.to_dict()

y_val_phys = np.array([
    y_val_anom[i] + monthly_mean_dict[meta_month_val[i]]
    for i in range(len(y_val_anom))
])

y_mean_pred_phys = np.array([
    y_mean_pred_anom[i] + monthly_mean_dict[meta_month_val[i]]
    for i in range(len(y_mean_pred_anom))
])

y_quantile_pred_phys = np.array([
    y_quantile_pred_anom[i] + monthly_mean_dict[meta_month_val[i]]
    for i in range(len(y_quantile_pred_anom))
])


# ============================================================
# Helper: full evaluation
# ============================================================
def evaluate_head(name, y_pred_phys):
    errors = y_val_phys - y_pred_phys
    abs_errors = np.abs(errors)

    mae = abs_errors.mean()
    rmse = np.sqrt(np.mean(errors ** 2))

    # July diagnostics
    july_mask = meta_month_val == 7
    july_mae = abs_errors[july_mask].mean() if july_mask.any() else np.nan

    rank_corr = np.nan
    if july_mask.sum() > 2:
        obs = y_val_phys[july_mask]
        pred = y_pred_phys[july_mask]
        obs_rank = np.argsort(np.argsort(obs))
        pred_rank = np.argsort(np.argsort(pred))
        rank_corr = np.corrcoef(obs_rank, pred_rank)[0, 1]

    # Extreme bias
    q90 = np.percentile(y_val_phys, 90)
    extreme_mask = y_val_phys >= q90
    extreme_bias = (
        np.mean(y_pred_phys[extreme_mask] - y_val_phys[extreme_mask])
        if extreme_mask.any() else np.nan
    )

    return {
        "MAE": mae,
        "RMSE": rmse,
        "July_MAE": july_mae,
        "July_Rank": rank_corr,
        "Extreme_Bias": extreme_bias
    }

# ============================================================
# Run evaluation
# ============================================================
mean_metrics = evaluate_head("MEAN", y_mean_pred_phys)
quantile_metrics = evaluate_head("QUANTILE", y_quantile_pred_phys)

# ============================================================
# SIDE-BY-SIDE SUMMARY
# ============================================================
print("\n================ TWO-HEAD COMPARISON =================")
print(f"{'Metric':<20} {'Mean Head':>12} {'Quantile Head':>15}")
print("-" * 50)

for key in mean_metrics:
    print(
        f"{key:<20} "
        f"{mean_metrics[key]:>12.3f} "
        f"{quantile_metrics[key]:>15.3f}"
    )

# ============================================================
# MAE BY HORIZON (OPTIONAL BUT IDENTICAL)
# ============================================================
print("\nMAE by Horizon (Mean vs Quantile):")
for h in sorted(np.unique(meta_horizon_val)):
    mask = meta_horizon_val == h
    mae_mean = np.abs(y_val_phys[mask] - y_mean_pred_phys[mask]).mean()
    mae_quant = np.abs(y_val_phys[mask] - y_quantile_pred_phys[mask]).mean()
    print(
        f"  h={h:2d} → Mean={mae_mean:.3f} | Quantile={mae_quant:.3f}"
    )


8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step

================ TWO-HEAD COMPARISON =================
Metric                  Mean Head   Quantile Head
--------------------------------------------------
MAE                         0.207           0.280
RMSE                        0.260           0.340
July_MAE                      nan             nan
July_Rank                     nan             nan
Extreme_Bias               -0.420          -0.119

MAE by Horizon (Mean vs Quantile):
  h= 1 → Mean=0.188 | Quantile=0.266
  h= 2 → Mean=0.198 | Quantile=0.260
  h= 3 → Mean=0.202 | Quantile=0.254
  h= 4 → Mean=0.194 | Quantile=0.269
  h= 5 → Mean=0.206 | Quantile=0.274
  h= 6 → Mean=0.214 | Quantile=0.289
  h= 7 → Mean=0.214 | Quantile=0.280
  h= 8 → Mean=0.215 | Quantile=0.288
  h= 9 → Mean=0.204 | Quantile=0.269
  h=10 → Mean=0.216 | Quantile=0.296
  h=11 → Mean=0.212 | Quantile=0.296
  h=12 → Mean=0.220 | Quantile=0.322


In [10]:
def evaluate_head(name, y_pred_phys):
    errors = y_val_phys - y_pred_phys
    abs_errors = np.abs(errors)

    mae = abs_errors.mean()
    rmse = np.sqrt(np.mean(errors ** 2))

    # July diagnostics
    july_mask = meta_month_val == 7
    july_mae = abs_errors[july_mask].mean()

    rank_corr = np.nan
    if july_mask.sum() > 2:
        obs = y_val_phys[july_mask]
        pred = y_pred_phys[july_mask]
        obs_rank = np.argsort(np.argsort(obs))
        pred_rank = np.argsort(np.argsort(pred))
        rank_corr = np.corrcoef(obs_rank, pred_rank)[0, 1]

    # Extreme bias
    q90 = np.percentile(y_val_phys, 90)
    extreme_mask = y_val_phys >= q90
    extreme_bias = np.mean(
        y_pred_phys[extreme_mask] - y_val_phys[extreme_mask]
    )

    return {
        "MAE": mae,
        "RMSE": rmse,
        "July_MAE": july_mae,
        "July_Rank": rank_corr,
        "Extreme_Bias": extreme_bias
    }


In [11]:
mean_metrics = evaluate_head("MEAN", y_mean_pred_phys)
quant_metrics = evaluate_head("QUANTILE", y_quantile_pred_phys)

print("\n================ TWO-HEAD COMPARISON =================")
print(f"{'Metric':<20} {'Mean Head':>12} {'Quantile Head':>15}")
print("-" * 50)

for k in mean_metrics:
    print(f"{k:<20} {mean_metrics[k]:>12.3f} {quant_metrics[k]:>15.3f}")



================ TWO-HEAD COMPARISON =================
Metric                  Mean Head   Quantile Head
--------------------------------------------------
MAE                         0.224           0.094
RMSE                        0.246           0.117
July_MAE                    0.224           0.094
July_Rank                  -0.061           0.212
Extreme_Bias               -0.298           0.023


## Ensemble

In [7]:
def evaluate_head(
    y_true_phys: np.ndarray,
    y_pred_phys: np.ndarray,
    meta_month_val: np.ndarray,
    meta_horizon_val: np.ndarray,
    *,
    focus_month: int | None = 7,
    enable_extreme_gate: bool = True,
    extreme_percentile: float = 90.0,
) -> dict:
    """
    Evaluate predictions in physical units.

    Args:
      y_true_phys, y_pred_phys: arrays shape (N,)
      meta_month_val, meta_horizon_val: arrays shape (N,)
      focus_month: month for month-specific diagnostics (None disables)
      enable_extreme_gate: on/off for extreme diagnostics
      extreme_percentile: percentile threshold computed on y_true_phys (global)

    Returns:
      dict of metrics comparable to earlier evaluation.
    """
    y_true_phys = np.asarray(y_true_phys).reshape(-1)
    y_pred_phys = np.asarray(y_pred_phys).reshape(-1)

    if y_true_phys.shape != y_pred_phys.shape:
        raise ValueError(f"Shape mismatch: y_true {y_true_phys.shape} vs y_pred {y_pred_phys.shape}")

    errors = y_true_phys - y_pred_phys
    abs_errors = np.abs(errors)

    mae = float(abs_errors.mean())
    rmse = float(np.sqrt(np.mean(errors ** 2)))

    # Month diagnostics (default July)
    month_mae = np.nan
    rank_corr = np.nan
    if focus_month is not None:
        m_mask = (meta_month_val == focus_month)
        if m_mask.any():
            month_mae = float(abs_errors[m_mask].mean())
            if m_mask.sum() > 2:
                obs = y_true_phys[m_mask]
                pred = y_pred_phys[m_mask]
                obs_rank = np.argsort(np.argsort(obs))
                pred_rank = np.argsort(np.argsort(pred))
                rank_corr = float(np.corrcoef(obs_rank, pred_rank)[0, 1])

    # Extreme diagnostics gate
    extreme_bias = np.nan
    if enable_extreme_gate:
        thr = np.percentile(y_true_phys, extreme_percentile)
        extreme_mask = y_true_phys >= thr
        if extreme_mask.any():
            extreme_bias = float(np.mean(y_pred_phys[extreme_mask] - y_true_phys[extreme_mask]))

    return {
        "MAE": mae,
        "RMSE": rmse,
        f"{'July' if focus_month==7 else 'Month'}_MAE": month_mae,
        f"{'July' if focus_month==7 else 'Month'}_Rank": rank_corr,
        "Extreme_Bias": extreme_bias,
    }

In [9]:
N_ENSEMBLE = 10
ensemble_results = []
monthly_mean_dict = monthly_mean.to_dict()
for k in range(N_ENSEMBLE):
    tf.keras.backend.clear_session()
    tf.random.set_seed(100 + k)
    np.random.seed(100 + k)
    print("ENSEMBLE MEMBER", k)
    model = build_two_head_nn(X_train.shape[1])
    model.fit(
        X_train,
        {"mean": y_train, "quantile": y_train},
        validation_data=(X_val, {"mean": y_val, "quantile": y_val}),
        epochs=20,
        callbacks=[EarlyStopping(patience=15, restore_best_weights=True)],
        verbose=0
    )

    pred_mean, pred_quantile = model.predict(X_val)

    pred_mean = pred_mean.ravel()
    pred_quantile = pred_quantile.ravel()

    mean_anom = y_scaler.inverse_transform(pred_mean[:, None]).ravel()
    quant_anom = y_scaler.inverse_transform(pred_quantile[:, None]).ravel()
    y_obs_anom = y_scaler.inverse_transform(y_val[:, None]).ravel()

    mean_phys = np.array([
        mean_anom[i] + monthly_mean_dict[meta_month_val[i]]
        for i in range(len(mean_anom))
    ])

    quant_phys = np.array([
        quant_anom[i] + monthly_mean_dict[meta_month_val[i]]
        for i in range(len(quant_anom))
    ])

    obs_phys = np.array([
        y_obs_anom[i] + monthly_mean_dict[meta_month_val[i]]
        for i in range(len(y_obs_anom))
    ])

    mean_metrics = evaluate_head(
        obs_phys, mean_phys,
        meta_month_val, meta_horizon_val,
        focus_month=7,
        enable_extreme_gate=True,
        extreme_percentile=90
    )

    quant_metrics = evaluate_head(
        obs_phys, quant_phys,
        meta_month_val, meta_horizon_val,
        focus_month=7,
        enable_extreme_gate=True,
        extreme_percentile=90
    )

    ensemble_results.append({"member": k, "head": "mean", **mean_metrics})
    ensemble_results.append({"member": k, "head": "quantile", **quant_metrics})


ENSEMBLE MEMBER 0


1/3 ━━━━━━━━━━━━━━━━━━━━ 0s 159ms/stepWARNING:tensorflow:6 out of the last 9 calls to <function TensorFlowTrainer.make_predict_function.<locals>.one_step_on_data_distributed at 0x7b37c03efba0> triggered tf.function retracing. Tracing is expensive and the excessive number of tracings could be due to (1) creating @tf.function repeatedly in a loop, (2) passing tensors with different shapes, (3) passing Python objects instead of tensors. For (1), please define your @tf.function outside of the loop. For (2), @tf.function has reduce_retracing=True option that can avoid unnecessary retracing. For (3), please refer to https://www.tensorflow.org/guide/function#controlling_retracing and https://www.tensorflow.org/api_docs/python/tf/function for  more details.


3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step
ENSEMBLE MEMBER 1
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
ENSEMBLE MEMBER 2
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step
ENSEMBLE MEMBER 3
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
ENSEMBLE MEMBER 4
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
ENSEMBLE MEMBER 5
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step
ENSEMBLE MEMBER 6
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step
ENSEMBLE MEMBER 7
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step
ENSEMBLE MEMBER 8
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
ENSEMBLE MEMBER 9
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step


In [10]:
ensemble_results

[{'member': 0,
  'head': 'mean',
  'MAE': 0.2383691966533661,
  'RMSE': 0.26047393679618835,
  'July_MAE': 0.2383691966533661,
  'July_Rank': -0.05879266193694638,
  'Extreme_Bias': -0.27917206287384033},
 {'member': 0,
  'head': 'quantile',
  'MAE': 0.09259501844644547,
  'RMSE': 0.11436363309621811,
  'July_MAE': 0.09259501844644547,
  'July_Rank': 0.11821502036018638,
  'Extreme_Bias': 0.03323899582028389},
 {'member': 1,
  'head': 'mean',
  'MAE': 0.23609007894992828,
  'RMSE': 0.2578151524066925,
  'July_MAE': 0.23609007894992828,
  'July_Rank': -0.1963813441920994,
  'Extreme_Bias': -0.28632310032844543},
 {'member': 1,
  'head': 'quantile',
  'MAE': 0.10823196172714233,
  'RMSE': 0.13461816310882568,
  'July_MAE': 0.10823196172714233,
  'July_Rank': -0.12690483187103815,
  'Extreme_Bias': -0.013088161125779152},
 {'member': 2,
  'head': 'mean',
  'MAE': 0.21946237981319427,
  'RMSE': 0.23796691000461578,
  'July_MAE': 0.21946237981319427,
  'July_Rank': -0.06647495906972839,
  '